# artificial-sort — Poset RL Demo

Learn to sort with the fewest pairwise comparisons using reinforcement learning.

**Works on CPU, GPU (CUDA), and Apple MPS** — device is detected automatically.  
To run on Google Colab with a free T4 GPU: *Runtime → Change runtime type → GPU*.

## 1 · Install and Import Dependencies

Install the package directly from GitHub (or from a local checkout when developing).

In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# ── install from GitHub (swap the comment to use a local path instead) ──
_pip("artificial-sort[notebook] @ git+https://github.com/blackgauss/artificial-sort.git@feat/attention-policy")
# _pip("-e", "..")   # ← uncomment when working from a local clone

import math, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import torch

print(f"PyTorch {torch.__version__}")

## 2 · Detect Available Hardware

In [ ]:
def detect_device() -> str:
    """Return the best available torch device string."""
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem  = torch.cuda.get_device_properties(0).total_memory / 2**30
        print(f"✓  CUDA GPU detected: {name}  ({mem:.1f} GiB)")
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("✓  Apple MPS detected")
        return "mps"
    print("·  No GPU found — using CPU")
    return "cpu"

DEVICE = detect_device()
print(f"Active device: {DEVICE}")

## 3 · Package Structure Overview

The `poset_rl` package is organised as follows — every public symbol is importable directly:

```
poset_rl/
  env.py           ← PosetEnv  (comparison oracle + transitive closure)
  model.py         ← ActorCritic  (MLP, fixed n)
  attention_model.py ← AttentionActorCritic  (Transformer, any n)
  datasets.py      ← UniformSampler, ZipfSampler
  train.py         ← train(), run_episode(), train_step()
  bench.py         ← benchmark(), evaluate_model(), evaluate_baseline()
```

In [ ]:
from poset_rl import PosetEnv, ActorCritic, AttentionActorCritic
from poset_rl.train import train
from poset_rl.bench import (
    agent_random, agent_greedy,
    evaluate_baseline, evaluate_model,
    benchmark, print_table,
)

# Quick sanity check: one random episode on n=5
perm = np.random.permutation(5)
rel  = PosetEnv.total_order_from_perm(perm)
env  = PosetEnv(rel)
env.reset()
done = False
while not done:
    mask = env.legal_actions_mask()
    action = agent_greedy(env)
    _, _, done, _ = env.step(action)
print(f"Greedy sorted 5 elements in {env.steps} comparisons "
      f"(lower bound = {math.ceil(math.log2(math.factorial(5)))}, "
      f"worst case = {5*4//2})")

## 4 · Configure Device Strategy

`train()` accepts an explicit `device` argument and moves the model and all tensors
to that device automatically.  On Colab T4 this gives a **~10–15× speedup** over CPU
for the Transformer model because each episode is still a single sample but the
backward pass through the Transformer layers dominates.

In [ ]:
# ── training hyper-parameters ── tweak these freely ──────────────────────
N          = 5          # fixed n for MLP training
NS_CURR    = [3,4,5,6]  # curriculum range for Attention model
TRAIN_EPS  = 3000       # training episodes
EVAL_EPS   = 300        # evaluation episodes per n
HIDDEN     = 64
NHEAD      = 2
NLAYERS    = 1
LR         = 3e-3
LOG_EVERY  = 500
SEED       = 42
# ─────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Will train on device='{DEVICE}'")

## 5 · Train the MLP Agent (fixed n)

`ActorCritic` uses a two-layer MLP that observes the full $n^2$ known-relation
matrix.  One model per $n$.

In [ ]:
mlp = ActorCritic(N, hidden=HIDDEN)

t0 = time.time()
mlp_history = train(
    mlp,
    n_or_range=N,
    episodes=TRAIN_EPS,
    lr=LR,
    device=DEVICE,
    log_interval=LOG_EVERY,
    out_csv="mlp_training.csv",
)
print(f"\nMLP trained in {time.time()-t0:.1f}s")

## 6 · Train the Attention Agent (curriculum over multiple n)

`AttentionActorCritic` processes *pair-level* features with a Transformer — no
positional encodings — so the same weights work for any $n$.
Training cycles through `NS_CURR` each episode.

In [ ]:
attn = AttentionActorCritic(hidden=HIDDEN, nhead=NHEAD, nlayers=NLAYERS)

t0 = time.time()
attn_history = train(
    attn,
    n_or_range=NS_CURR,
    episodes=TRAIN_EPS,
    lr=LR,
    device=DEVICE,
    log_interval=LOG_EVERY,
    out_csv="attn_training.csv",
)
print(f"\nAttention model trained in {time.time()-t0:.1f}s")

## 7 · Benchmark and Visualise

In [ ]:
# ── evaluate all agents on a shared test set ─────────────────────────────
BENCH_NS = sorted(set(NS_CURR) | {N})

print("Evaluating baselines …")
rand_res   = evaluate_baseline(agent_random, BENCH_NS, EVAL_EPS)
greedy_res = evaluate_baseline(agent_greedy, BENCH_NS, EVAL_EPS)

print("Evaluating MLP …")
mlp_res = evaluate_model(mlp, [N], EVAL_EPS, device=DEVICE)

print("Evaluating Attention …")
attn_res = evaluate_model(attn, BENCH_NS, EVAL_EPS, device=DEVICE)

# ── table ─────────────────────────────────────────────────────────────────
print(f"\n{'n':>3}  {'lb':>4}  {'worst':>5}  {'random':>8}  {'greedy':>8}"
      f"  {'mlp':>8}  {'attention':>9}")
print("-" * 58)
for n in BENCH_NS:
    lb    = math.ceil(math.log2(math.factorial(n)))
    worst = n * (n-1) // 2
    mlp_v = f"{mlp_res[n]:8.2f}" if n in mlp_res else "       —"
    print(f"{n:>3}  {lb:>4}  {worst:>5}  {rand_res[n]:8.2f}  "
          f"{greedy_res[n]:8.2f}  {mlp_v}  {attn_res[n]:9.2f}")
print("\nlb = ⌈log₂(n!)⌉ (information-theoretic minimum)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── left: learning curves ──────────────────────────────────────────────
ax = axes[0]
window = 100

def _smooth(vals, w):
    return np.convolve(vals, np.ones(w)/w, mode="valid")

mlp_steps  = [r["steps"] for r in mlp_history]
attn_steps = [r["steps"] for r in attn_history]

ax.plot(_smooth(mlp_steps,  window), label=f"MLP (n={N})",         color="steelblue")
ax.plot(_smooth(attn_steps, window), label=f"Attention (n={NS_CURR})", color="tomato")
ax.axhline(math.ceil(math.log2(math.factorial(N))),
           color="steelblue", linestyle="--", linewidth=0.8, label=f"lb n={N}")
ax.set_xlabel("Episode")
ax.set_ylabel("Comparisons")
ax.set_title("Learning curves (smoothed)")
ax.legend()
ax.grid(True, alpha=0.3)

# ── right: bar chart per-n ─────────────────────────────────────────────
ax = axes[1]
x  = np.arange(len(BENCH_NS))
w  = 0.18
lb_vals = [math.ceil(math.log2(math.factorial(n))) for n in BENCH_NS]

ax.bar(x - 1.5*w, lb_vals,                             w, label="lower bound", color="gold",      alpha=0.8)
ax.bar(x - 0.5*w, [rand_res[n]   for n in BENCH_NS],   w, label="random",      color="lightgray", alpha=0.9)
ax.bar(x + 0.5*w, [greedy_res[n] for n in BENCH_NS],   w, label="greedy",      color="mediumseagreen", alpha=0.9)
ax.bar(x + 1.5*w, [attn_res.get(n, float("nan")) for n in BENCH_NS],
                                                        w, label="attention",   color="tomato",    alpha=0.9)
# MLP only on its n
mlp_x = BENCH_NS.index(N)
ax.bar(mlp_x,     mlp_res.get(N, float("nan")),         w, label="mlp",         color="steelblue", alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels([f"n={n}" for n in BENCH_NS])
ax.set_ylabel("Mean comparisons")
ax.set_title("Agent comparison per n")
ax.legend(fontsize=8)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("benchmark.png", dpi=150)
plt.show()
print("Saved benchmark.png")